# Aiscern Image Detector — Fine-Tuning
**Platform:** Kaggle (T4 GPU)  
**Target:** `saghi776/aiscern-image-detector`  
**Base model:** `google/vit-base-patch16-224-in21k`  
**Expected accuracy:** >88%  
**Runtime:** ~45 minutes

### Before running:
1. Set **Accelerator → GPU T4 x2** in notebook settings
2. Add secret `HF_TOKEN` in Kaggle → Add-ons → Secrets
3. Create HF repo first: https://huggingface.co/new?owner=saghi776&name=aiscern-image-detector

In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers datasets accelerate huggingface_hub pillow scikit-learn evaluate

In [ ]:
# CELL 2 — Authenticate HuggingFace
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets   = UserSecretsClient()
HF_TOKEN  = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ HuggingFace authenticated')

In [ ]:
# CELL 3 — Configuration
MODEL_ID    = 'google/vit-base-patch16-224-in21k'
REPO_ID     = 'saghi776/aiscern-image-detector'
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 2e-5
SEED        = 42
MAX_SAMPLES = 20000   # 10K AI + 10K real — fits in 45 min
IMG_SIZE    = 224

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 4 — Load CIFAKE dataset (clean, balanced, pre-labeled)
# CIFAKE: 60K AI-generated (Stable Diffusion) + 60K real (CIFAR-10)
# Labels in dataset: 'FAKE' = AI generated, 'REAL' = real photograph
from datasets import load_dataset, concatenate_datasets
import numpy as np

print('Loading CIFAKE dataset...')
cifake_train = load_dataset('jlbaker361/cifake-real-and-ai-generated-small-images', split='train')
cifake_test  = load_dataset('jlbaker361/cifake-real-and-ai-generated-small-images', split='test')

print(f'CIFAKE train: {len(cifake_train):,} samples')
print(f'CIFAKE test:  {len(cifake_test):,} samples')
print(f'Columns: {cifake_train.column_names}')
print(f'Label example: {cifake_train[0]["label"]}')

In [ ]:
# CELL 5 — Normalize labels to 0=REAL, 1=AI_GENERATED
# CIFAKE uses integer labels — check and remap
from datasets import ClassLabel

def normalize_labels(dataset):
    """Map any label format to 0=REAL, 1=AI_GENERATED"""
    def map_label(example):
        lbl = example['label']
        if isinstance(lbl, str):
            lbl_lower = lbl.lower()
            if any(x in lbl_lower for x in ['fake', 'ai', 'generated', 'synthetic', '1']):
                example['label'] = 1
            else:
                example['label'] = 0
        # If already int: CIFAKE uses 0=REAL, 1=FAKE — correct
        return example
    return dataset.map(map_label)

cifake_train = normalize_labels(cifake_train)
cifake_test  = normalize_labels(cifake_test)

# Balance and cap for time budget
half = MAX_SAMPLES // 2
real_train = cifake_train.filter(lambda x: x['label'] == 0).shuffle(SEED).select(range(min(half, len(cifake_train.filter(lambda x: x['label'] == 0)))))
ai_train   = cifake_train.filter(lambda x: x['label'] == 1).shuffle(SEED).select(range(min(half, len(cifake_train.filter(lambda x: x['label'] == 1)))))
balanced   = concatenate_datasets([real_train, ai_train]).shuffle(SEED)

# Train/val split (90/10)
split      = balanced.train_test_split(test_size=0.1, seed=SEED)
train_ds   = split['train']
val_ds     = split['test']
test_ds    = cifake_test

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')
real_count = sum(1 for x in train_ds if x['label'] == 0)
ai_count   = len(train_ds) - real_count
print(f'Train balance — Real: {real_count:,}  AI: {ai_count:,}')

In [ ]:
# CELL 6 — Image preprocessing with ViT processor
from transformers import ViTImageProcessor
from PIL import Image

processor = ViTImageProcessor.from_pretrained(MODEL_ID)

def preprocess(examples):
    images = []
    for img in examples['image']:
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        images.append(img.convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
    inputs = processor(images=images, return_tensors='pt')
    inputs['labels'] = examples['label']
    return inputs

print('Preprocessing train set...')
train_ds = train_ds.map(preprocess, batched=True, batch_size=256,
                         remove_columns=['image'], desc='Train preprocessing')
print('Preprocessing val set...')
val_ds   = val_ds.map(preprocess, batched=True, batch_size=256,
                       remove_columns=['image'], desc='Val preprocessing')
print('Preprocessing test set...')
test_ds  = test_ds.map(preprocess, batched=True, batch_size=256,
                        remove_columns=['image'], desc='Test preprocessing')

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('✅ Preprocessing complete')

In [ ]:
# CELL 7 — Load ViT model for binary classification
from transformers import ViTForImageClassification

# CRITICAL: id2label must use AI-pattern words for hf-analyze.ts compatibility
# hf-analyze.ts regex: /ai|fake|generated|artificial/i → matches 'AI_GENERATED'
# hf-analyze.ts regex: /real|authentic|genuine/i       → matches 'REAL'
model = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: 'REAL', 1: 'AI_GENERATED'},
    label2id={'REAL': 0, 'AI_GENERATED': 1},
    ignore_mismatched_sizes=True,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total_params / 1e6:.1f}M')
print(f'Trainable params: {trainable_params / 1e6:.1f}M')

In [ ]:
# CELL 8 — Training arguments
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

training_args = TrainingArguments(
    output_dir                  = './aiscern-image-detector',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    logging_steps               = 50,
    fp16                        = True,       # T4 supports FP16
    dataloader_num_workers      = 2,
    push_to_hub                 = True,
    hub_model_id                = REPO_ID,
    hub_token                   = HF_TOKEN,
    hub_strategy                = 'every_save',
    report_to                   = 'none',
    seed                        = SEED,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='binary')),
    }

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print('✅ Trainer ready')

In [ ]:
# CELL 9 — Train
print('Starting training...')
train_result = trainer.train()
print(f'Training complete!')
print(f'Total steps:   {train_result.global_step}')
print(f'Training loss: {train_result.training_loss:.4f}')

In [ ]:
# CELL 10 — Evaluate on held-out test set
print('Evaluating on test set...')
test_results = trainer.evaluate(test_ds)

print(f"\n{'='*40}")
print(f"TEST RESULTS")
print(f"{'='*40}")
print(f"Accuracy: {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.1f}%)")
print(f"F1 Score: {test_results['eval_f1']:.4f}")
print(f"Loss:     {test_results['eval_loss']:.4f}")
target = 0.88
status = '✅ TARGET MET' if test_results['eval_accuracy'] >= target else f'⚠️  Below {target:.0%} target'
print(f"\nTarget (>88%): {status}")

In [ ]:
# CELL 11 — Push to HuggingFace Hub
acc = test_results['eval_accuracy']
f1  = test_results['eval_f1']

trainer.push_to_hub(
    commit_message=f'Fine-tuned ViT — acc={acc:.3f} f1={f1:.3f} | Aiscern image detector'
)
processor.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'✅ Model pushed to https://huggingface.co/{REPO_ID}')

In [ ]:
# CELL 12 — Verify inference API output format
# hf-analyze.ts expects: [{label: 'AI_GENERATED', score: 0.92}, {label: 'REAL', score: 0.08}]
import time
print('Waiting 60s for HF to load the model...')
time.sleep(60)

from transformers import pipeline
from PIL import Image

pipe = pipeline('image-classification', model=REPO_ID, token=HF_TOKEN)

# Test with an AI sample from test set
ai_samples   = [test_ds[i] for i in range(len(test_ds)) if test_ds[i]['labels'] == 1][:3]
real_samples = [test_ds[i] for i in range(len(test_ds)) if test_ds[i]['labels'] == 0][:3]

print('\n--- AI samples (expected: AI_GENERATED > 0.5) ---')
for s in ai_samples:
    # Reconstruct PIL image from pixel values
    import torchvision.transforms as T
    img = T.ToPILImage()(s['pixel_values'])
    result = pipe(img)
    top = result[0]
    ok = '✅' if top['label'] == 'AI_GENERATED' else '❌'
    print(f'  {ok} {top["label"]}: {top["score"]:.3f}')

print('\n--- Real samples (expected: REAL > 0.5) ---')
for s in real_samples:
    img = T.ToPILImage()(s['pixel_values'])
    result = pipe(img)
    top = result[0]
    ok = '✅' if top['label'] == 'REAL' else '❌'
    print(f'  {ok} {top["label"]}: {top["score"]:.3f}')

print(f'\nFull output format: {pipe(T.ToPILImage()(ai_samples[0]["pixel_values"]))[:]}')
print('\n✅ Model is live and compatible with hf-analyze.ts')